# Alpamayo-v1.5 Phase 3b — Random VLM-weight BFA (re-prefill baseline)

Random-coord baseline for the gradient-guided sweep in `bfa_vlm_grad.ipynb`. Same module subsample, same trial count, same per-trial cost — only the coord-selection rule differs (`random_topk_coords` vs. `topk_bitflip_coords`). Used to quantify how much the gradient-guided ranking actually buys vs. uniform sampling.

**No gradients are computed in this notebook** — the VLM backward step is what makes Phase 3a expensive at startup; Phase 3b skips it. Per-trial cost is unchanged (~650 ms re-prefill) because the measurement is identical.

**Shared-GPU note.** Set `GPU_ID` below to your allocated GPU. All CUDA placement and `bfa.*` calls take `DEVICE` explicitly — no silent `cuda:0` fallback.

16 bits × 50 modules × 5 coords ≈ 4k trials → ~45 min/scene on one H20.
Output: `results/bfa_vlm_random_<scene>.parquet`.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo1_5 import helper

import bfa_utils as bfa

In [ ]:
GPU_ID = int(os.environ.get("ALPAMAYO_GPU", "0"))
DEVICE = torch.device(f"cuda:{GPU_ID}")
print(f"Using device: {DEVICE} (visible CUDA devices: {torch.cuda.device_count()})")

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to(DEVICE)
model.eval()
# requires_grad isn't needed here, but leaving the default doesn't hurt and
# keeps the model state identical to Phase 3a so smoke-test parity holds.

processor = helper.get_processor(model.tokenizer)

clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
SCENE_IDX = 774
clip_id = clip_ids[SCENE_IDX]
data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(
    data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    DEVICE,
)

In [ ]:
gt_action = model.action_space.traj_to_action(
    traj_history_xyz=data["ego_history_xyz"].to(DEVICE).to(torch.float32),
    traj_history_rot=data["ego_history_rot"].to(DEVICE).to(torch.float32),
    traj_future_xyz=data["ego_future_xyz"].to(DEVICE).to(torch.float32),
    traj_future_rot=data["ego_future_rot"].to(DEVICE).to(torch.float32),
)
gt_action = gt_action.view(-1, *model.action_space.get_action_space_dims())

with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
    ctx = bfa.build_fm_context(model, model_inputs, gt_action)

with torch.cuda.device(DEVICE):
    torch.cuda.manual_seed(42)
fixed_noise = torch.randn_like(ctx.gt_action)
T_VAL = 0.5

def loss_fn():
    with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
        return bfa.fm_one_step_loss(model, ctx, t_val=T_VAL, noise=fixed_noise)

baseline = loss_fn().item()
print("baseline FM loss:", baseline)

In [ ]:
# Same module subsample as Phase 3a: deterministic via numpy seed=0.
all_targets = bfa.collect_target_linears_vlm(model)
MODULE_K = 50
rng = np.random.default_rng(0)
names_sorted = sorted(all_targets.keys())
subsample_idx = rng.choice(len(names_sorted), size=min(MODULE_K, len(names_sorted)), replace=False)
targets = {names_sorted[i]: all_targets[names_sorted[i]] for i in sorted(subsample_idx)}
print(f"targets: {len(targets)} (subsample matches bfa_vlm_grad.ipynb)")
print("first 5:", list(targets.keys())[:5])

## Smoke tests

Phase 3b skips smoke tests (a) and (b) — no backward to validate, no peak-memory budget to check. (c)–(e) below verify the re-prefill measurement and restore round-trip.

In [ ]:
# (c) Mantissa-LSB sanity via the re-prefill path.
name = list(targets.keys())[0]
mod = targets[name]
rng_smoke = np.random.default_rng(1)
rand_idx = int(rng_smoke.integers(0, mod.weight.numel()))

r = bfa.measure_vlm_flipped_loss_reprefill(
    model, model_inputs, gt_action,
    module=mod, flat_idx=rand_idx, bit=0,
    noise=fixed_noise, device=DEVICE, t_val=T_VAL,
)
delta = r["post_loss"] - baseline
print(f"mantissa LSB Δloss: {delta:.4g}")
assert r["post_loss_finite"] == 1.0

# (e) Re-prefill consistency: cache-stable loss must match baseline after restore.
post_restore = loss_fn().item()
assert abs(post_restore - baseline) < 1e-5, (post_restore, baseline)
print(f"restore + re-prefill consistency: OK ({post_restore})")

In [ ]:
# (d) Catastrophic exponent: bit 14 should produce inf/nan loss for some coords.
r_bad = bfa.measure_vlm_flipped_loss_reprefill(
    model, model_inputs, gt_action,
    module=mod, flat_idx=rand_idx, bit=14,
    noise=fixed_noise, device=DEVICE, t_val=T_VAL,
)
print(f"bit-14 post_loss: {r_bad['post_loss']:.4g}, finite={r_bad['post_loss_finite']}")
post_restore = loss_fn().item()
assert abs(post_restore - baseline) < 1e-5
print("catastrophic-bit restore: OK")

## Main sweep

Per-trial cost dominated by the in-trial VLM re-prefill (~600 ms) — same as Phase 3a. Difference is only the coord selection: `random_topk_coords` instead of `topk_bitflip_coords`.

In [ ]:
BITS = list(range(16))
TOP_K_PER_MODULE = 5
MODULE_SUBSET = list(targets.keys())
RNG_RANDOM = np.random.default_rng(2026)  # distinct seed from module-subsample rng

results = []
t_start = time.time()
for bit in BITS:
    for name in MODULE_SUBSET:
        mod = targets[name]
        # Fresh random sample per (bit, module) — matches Phase 3a's per-(bit,module) k=5 structure.
        flat_idx, _ = bfa.random_topk_coords(mod.weight, bit=bit, k=TOP_K_PER_MODULE, rng=RNG_RANDOM)
        for i, idx in enumerate(flat_idx.tolist()):
            r = bfa.measure_vlm_flipped_loss_reprefill(
                model, model_inputs, gt_action,
                module=mod, flat_idx=idx, bit=bit,
                noise=fixed_noise, device=DEVICE, t_val=T_VAL,
            )
            r.update({
                "module": name,
                "rank": i,
                "predicted_bit_grad": float("nan"),  # no grad ranking
                "baseline_loss": baseline,
                "delta_loss": r["post_loss"] - baseline,
                "ranking": "random",
            })
            results.append(r)
    print(f"bit {bit:2d} done — {len(results):5d} trials, {time.time()-t_start:.0f}s elapsed")

In [ ]:
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"bfa_vlm_random_scene{SCENE_IDX}.parquet"
df = pd.DataFrame(results)
df.to_parquet(out_path)
print(f"saved {len(df)} trials to {out_path}")

In [ ]:
print(df.groupby("bit")["delta_loss"].describe()[["count", "mean", "max"]])
df_finite = df[df["post_loss_finite"] == 1.0]
print("\nTop-10 random flips by delta_loss (finite-loss only):")
print(df_finite.nlargest(10, "delta_loss")[["module", "bit", "flat_idx", "delta_loss"]])

fig, ax = plt.subplots(figsize=(10, 4))
df_finite.boxplot(column="delta_loss", by="bit", ax=ax, showfliers=False)
ax.set_title("Phase 3b (random) — Δloss by bit (finite-loss only)")
ax.set_ylabel("Δ FM loss")
plt.suptitle("")
plt.tight_layout()
plt.savefig(out_dir / f"bfa_vlm_random_per_bit_scene{SCENE_IDX}.png", dpi=120)
plt.show()